In [ ]:
import torch
import matplotlib.pyplot as plt
import einops
from torchvision.transforms.functional import adjust_contrast, adjust_brightness
from PIL import Image
import numpy as np

In [ ]:
!ls *.pt

In [ ]:
data = torch.load("test_aug_smpt.pt", map_location="cpu", weights_only=True)

In [ ]:
data.keys()

In [ ]:
data["region_crops"][0].shape

In [ ]:
crops = [
    einops.rearrange(adjust_brightness(adjust_contrast(c, 2), 3), "b c h w -> b h w c")
    for c in data["region_crops"]]


In [ ]:
layout = [
    ["g0","g0","g1","g1","l0","l1","l2","l3"],
    ["g0","g0","g1","g1","l4","l5","l6","l7"],
]

for i in range(20):
    plt.figure()
    
    fig, axs = plt.subplot_mosaic(layout, figsize=(16, 4), constrained_layout=True)
    axs["g0"].imshow(crops[0][i])
    axs["g1"].imshow(crops[1][i])

    for j in range(8):
        axs[f"l{j}"].imshow(crops[j+2][i])

    for ax in axs:
        axs[ax].axis("off")

In [ ]:
region_crops = [
    adjust_brightness(adjust_contrast(i, 2), 3)
    for i in data["region_crops"]
]

In [ ]:
data = torch.load("test_aug_ncc.pt", map_location="cpu")

In [ ]:
orig_im = einops.rearrange(adjust_brightness(adjust_contrast(data["orig_im"], 2), 3).squeeze(), "b c h w -> b h w c")

In [ ]:
crops = [einops.rearrange(i, "(b nh nw) c rh rw -> b (nh rh) (nw rw) c",  nh=6, nw=6) for i in region_crops]

In [ ]:
Image.fromarray((orig_im[6].numpy() * 255).astype(np.uint8)).save("noncc.png")

In [ ]:
fig, axs = plt.subplots(1,8, figsize=(16,2))
for a,i in zip(axs, orig_im):
    a.imshow(i)
    a.axis("off")

In [ ]:
for c in crops:
    fig, axs = plt.subplots(1,8, figsize=(16,2))
    for a,i in zip(axs, c):
        a.imshow(i)
        a.axis("off")

In [ ]:
for c in data["tile_masks"]:
    fig, axs = plt.subplots(1,8, figsize=(16,2))
    for a,i in zip(axs, c):
        a.imshow(i)
        a.axis("off")

In [ ]:
data = torch.load("stage1_out.pt", map_location="cpu", weights_only=True)

In [ ]:
plt.imshow(data["ss"][1,0,...].detach().numpy())

In [ ]:
plt.imshow(data["ss"][5,0,...].detach().numpy())

In [ ]:
data["ss"].shape

In [ ]:
def crop_different(x, h0, w0):
    # x: (b, c, h, w)
    b, c, h, w = x.shape
    device = x.device

    # Random top and left coordinates for each image in the batch
    top = torch.randint(0, h - h0 + 1, (b,), device=device)
    left = torch.randint(0, w - w0 + 1, (b,), device=device)

    # Create a range for the crop dimensions and expand it to (b, h0, w0)
    row_offsets = torch.arange(h0, device=device).view(1, h0, 1)
    col_offsets = torch.arange(w0, device=device).view(1, 1, w0)
    
    # Compute absolute row and column indices for each crop
    # top and left are reshaped to (b, 1, 1) so they broadcast properly
    rows = top.view(b, 1, 1) + row_offsets  # shape: (b, h0, 1)
    cols = left.view(b, 1, 1) + col_offsets   # shape: (b, 1, w0)
    
    # Create a batch index for advanced indexing
    batch_idx = torch.arange(b, device=device).view(b, 1, 1)
    
    # Use advanced indexing to extract the crop for each image.
    # This returns a tensor of shape (b, h0, w0, c)
    cropped = x[batch_idx, :, rows, cols]
    
    return cropped.permute(0, 3, 1, 2)


In [ ]:
ssc = crop_different(data["ss"], 5, 5)

In [ ]:
plt.imshow(ssc[0,..., 0].detach().numpy())

In [ ]:
plt.imshow(ssc[5,..., 0].detach().numpy())

In [ ]:
ssc.shape

In [ ]:
h0=w0=5

In [ ]:
x = data["ss"]

In [ ]:
b, c, h, w = x.shape
device = x.device

# Random top and left coordinates for each image in the batch
top = torch.randint(0, h - h0 + 1, (b,), device=device)
left = torch.randint(0, w - w0 + 1, (b,), device=device)

# Create a range for the crop dimensions and expand it to (b, h0, w0)
row_offsets = torch.arange(h0, device=device).view(1, h0, 1)
col_offsets = torch.arange(w0, device=device).view(1, 1, w0)


In [ ]:
col_offsets

In [ ]:

# Compute absolute row and column indices for each crop
# top and left are reshaped to (b, 1, 1) so they broadcast properly
rows = top.view(b, 1, 1) + row_offsets  # shape: (b, h0, 1)
cols = left.view(b, 1, 1) + col_offsets   # shape: (b, 1, w0)

In [ ]:

# Create a batch index for advanced indexing
batch_idx = torch.arange(b, device=device).view(b, 1, 1)


In [ ]:
batch_idx

In [ ]:

# Use advanced indexing to extract the crop for each image.
# This returns a tensor of shape (b, c, h0, w0)
cropped = x[batch_idx, :, rows, cols]


In [ ]:
x.shape

In [ ]:
cropped.shape

In [ ]:
x.shape

In [ ]:
batch_idx.shape

In [ ]:
rows.shape

In [ ]:
cols.shape


In [ ]:
import torch

In [ ]:
test = torch.load("/nfs/mm-isilon/brainscans/dropbox/exp/models/dinov2_github/dinov2_vits14_reg4_pretrain.pth")

In [ ]:
test["encoder.layer.0.attention.attention.query.weight"].shape

In [ ]:
test.keys()

In [ ]:
data["collated_masks"].sum(dim=1)

In [ ]:
data["n_masked_patches"]

In [ ]:
data["mask_indices_list"]

In [ ]:
data = torch.load("dinov2_fmi.pt", map_location="cpu", weights_only=True)

In [ ]:
data.keys()

In [ ]:
data["collated_global_crops"].shape

In [ ]:
data["collated_global_crops"][0].min()

In [ ]:
test = einops.rearrange(data["collated_global_crops"], "b c h w -> b h w c")

In [ ]:
test2 = test# test * torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])

In [ ]:
for i in range(50):
    plt.figure()
    plt.imshow(test2[i,...])

In [ ]:
for i in range(10):
    plt.figure()
    plt.imshow(test2[-i,...])

In [ ]:
plt.imshow(test[10,...,0])

In [ ]:
import torch
import matplotlib.pyplot as plt
import einops
from torchvision.transforms.functional import adjust_contrast, adjust_brightness
from PIL import Image
import numpy as np

In [ ]:
data = torch.load("dinov2_fmi.pt", map_location="cpu", weights_only=True)

In [ ]:
#from ts2.lm.mcm_dinov2_systems import MCMDinov2System
from dinov2.models import build_model_from_cfg
#import dinov2.utils.utils as dinov2_utils
from dinov2.layers import DINOHead

from urllib.parse import urlparse
import yaml
from omegaconf import OmegaConf

In [ ]:
def load_pretrained_weights(model, pretrained_weights, checkpoint_key):
    if urlparse(pretrained_weights).scheme:  # If it looks like an URL
        state_dict = torch.hub.load_state_dict_from_url(pretrained_weights,
                                                        map_location="cpu")
    else:
        state_dict = torch.load(pretrained_weights, map_location="cpu")
    if checkpoint_key is not None and checkpoint_key in state_dict:
        print(f"Take key {checkpoint_key} in provided checkpoint dict")
        state_dict = state_dict[checkpoint_key]
    # remove `module.` prefix
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    # remove `backbone.` prefix induced by multicrop wrapper
    state_dict = {k.replace("backbone.", ""): v for k, v in state_dict.items()}
    msg = model.load_state_dict(state_dict, strict=False)
    print(
        "Pretrained weights found at {} and loaded with msg: {}".format(
            pretrained_weights, msg))

In [ ]:
!ls /nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/96e19ab4_Apr11-01-59-24_sd1000_dev_tune0/models/eval/

In [ ]:
cf = OmegaConf.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/96e19ab4_Apr11-01-59-24_sd1000_dev_tune0/config/parsed_fair_config.json")
    

In [ ]:
ckpt_path = "/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/96e19ab4_Apr11-01-59-24_sd1000_dev_tune0/models/eval/training_16249/teacher_checkpoint.pth"
model, _ = build_model_from_cfg(cf.dinov2_fair_config, only_teacher=True)
load_pretrained_weights(model, ckpt_path, "teacher")
ckpt = torch.load(ckpt_path, map_location="cpu")["teacher"]

dino_head = DINOHead(
    in_dim=384,
    out_dim=cf.dinov2_fair_config.dino.head_n_prototypes,
    hidden_dim=cf.dinov2_fair_config.dino.head_hidden_dim,
    bottleneck_dim=cf.dinov2_fair_config.dino.head_bottleneck_dim,
    nlayers=cf.dinov2_fair_config.dino.head_nlayers
            )
dino_head.load_state_dict({i.removeprefix("dino_head."):ckpt[i] for i in ckpt if i.startswith("dino_head.")})

ibot_head = DINOHead(
    in_dim=384,
    out_dim=cf.dinov2_fair_config.ibot.head_n_prototypes,
    hidden_dim=cf.dinov2_fair_config.ibot.head_hidden_dim,
    bottleneck_dim=cf.dinov2_fair_config.ibot.head_bottleneck_dim,
    nlayers=cf.dinov2_fair_config.ibot.head_nlayers
            )
ibot_head.load_state_dict({i.removeprefix("ibot_head."):ckpt[i] for i in ckpt if i.startswith("ibot_head.")})

dino_head.cuda()
ibot_head.cuda()
model.cuda()
print("ok")

In [ ]:
test = model(data["collated_global_crops"].cuda().float())
iho = ibot_head(test).detach().cpu()
dho = dino_head(test).detach().cpu()
test = test.detach().cpu()

In [ ]:
print(torch.linalg.matrix_rank(test))
print(torch.linalg.matrix_rank(iho))
print(torch.linalg.matrix_rank(dho))

In [ ]:
iho.shape

In [ ]:
out = lm.tile_encoder.teacher.backbone(data["collated_global_crops"].cuda())

In [ ]:
out = lm.tile_encoder.student.backbone(data["collated_global_crops"].cuda())

In [ ]:
out.std(dim=1)

In [ ]:
del lm

In [ ]:
(out-0.0023).abs().sum(dim=1).min()

In [ ]:
lm.tile_encoder.teacher.dino_head

In [ ]:
del model,ibot_head,dino_head,test,iho,dho

torch.cuda.empty_cache()